# Notebook 3 — The Central Dogma in Code
## From DNA to RNA to Protein — with Real Biology

**Course:** Introduction to Bioinformatics — HCT Cohort

---

### What you'll do
You'll trace a real gene (BRCA1) through the **central dogma**:

**DNA → RNA → Protein**

By the end you will:
- Transcribe and translate sequences yourself
- Find open reading frames (ORFs)
- Detect mutations between two versions of a gene
- Distinguish synonymous (silent) from non-synonymous mutations

### Setup
Re-install Biopython if you're starting fresh.


In [ ]:
!pip install biopython --quiet
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction
print("Biopython ready")


---
## Exercise 1 — Load a real gene
We'll use a partial coding sequence from **BRCA1** — a real human gene where mutations are linked to breast cancer.


In [ ]:
brca1 = Seq("ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAAATCTTAGAGTGTCCCATCTGTCTGGAGTTGATCAAGGAACCTGTCTCCACAAAGTGTGACCACATATTTTGCAAATTTTGCATGCTGAAAC")
print("Length:    ", len(brca1))
print("First 30:  ", brca1[:30])
print("Last 30:   ", brca1[-30:])


---
## Exercise 2 — Composition
Let's check the base composition.


In [ ]:
print("A:", brca1.count("A"))
print("T:", brca1.count("T"))
print("G:", brca1.count("G"))
print("C:", brca1.count("C"))
print("GC content:", round(gc_fraction(brca1) * 100, 2), "%")


---
## Exercise 3 — Step 1 of the central dogma: Transcription
Transcribe DNA into RNA.


In [ ]:
brca1_rna = brca1.transcribe()
print("RNA (first 60):", brca1_rna[:60])


---
## Exercise 4 — Verify: every T became U
Check this transformation actually happened.


In [ ]:
print("T in DNA:", brca1.count("T"))
print("U in RNA:", brca1_rna.count("U"))
print("T in RNA:", brca1_rna.count("T"), "(should be 0)")
print("A, G, C unchanged?")
for base in ["A", "G", "C"]:
    dna_count = brca1.count(base)
    rna_count = brca1_rna.count(base)
    print(f"  {base}: DNA={dna_count}, RNA={rna_count}")


---
## Exercise 5 — The genetic code
Every 3 RNA bases (a codon) codes for 1 amino acid.
Biopython has a built-in codon table.


In [ ]:
from Bio.Data.CodonTable import standard_rna_table

# Show a few examples
examples = ["AUG", "GAU", "UUA", "UCU", "GCU", "UAA"]
for codon in examples:
    if codon in standard_rna_table.forward_table:
        aa = standard_rna_table.forward_table[codon]
        print(f"{codon} → {aa}")
    else:
        print(f"{codon} → STOP")


---
## Exercise 6 — Step 2 of the central dogma: Translation
Let Biopython translate the full RNA into protein.


In [ ]:
brca1_protein = brca1.translate()
print("Protein:   ", brca1_protein)
print("Length:    ", len(brca1_protein), "amino acids")


---
## Exercise 7 — Inspect the protein
Each letter is one amino acid. The first is `M` (Methionine), which is always the start.


In [ ]:
print("First amino acid:", brca1_protein[0])
print("Next 5:          ", brca1_protein[1:6])
print("Has stop codon?  ", "*" in brca1_protein)


---
## Exercise 8 — Find the start codon
The start codon is `ATG` in DNA (or `AUG` in RNA). Every protein begins here.


In [ ]:
start_pos = str(brca1).find("ATG")
print("First ATG at position:", start_pos)


### ✏️ Your Turn
Find the position of the SECOND `ATG` in BRCA1.


In [ ]:
# Hint: use .find() with a second argument to start the search past position 0
second_atg = str(brca1).find("ATG", ____)
print("Second ATG at position:", second_atg)


---
## Exercise 9 — Find stop codons
The three stop codons in DNA are `TAA`, `TAG`, `TGA`. When a ribosome hits one, it stops making protein.


In [ ]:
stops = ["TAA", "TAG", "TGA"]
for stop in stops:
    pos = str(brca1).find(stop)
    print(f"{stop}: first at position {pos}")


---
## Exercise 10 — Open Reading Frames (ORFs)
An **ORF** is a stretch from a start codon to the next in-frame stop codon.
That's what makes a protein.

Biopython can translate to a stop codon automatically.


In [ ]:
# Translate only until the first stop codon
protein_to_stop = brca1.translate(to_stop=True)
print("Protein (stops at first *):", protein_to_stop)
print("Length:                    ", len(protein_to_stop))


---
## Exercise 11 — Compare wild-type and mutant
A **mutation** is a single-base change. But not every mutation changes the protein — thanks to the redundancy of the genetic code.

Below we have two versions of a short sequence. One has a mutation. Let's see what it does.


In [ ]:
wild_type = Seq("ATGGATTTA")
mutant_1  = Seq("ATGGACTTA")  # 6th base: T → C

print("Wild-type DNA:    ", wild_type)
print("Mutant 1 DNA:     ", mutant_1)
print()
print("Wild-type protein:", wild_type.translate())
print("Mutant 1 protein: ", mutant_1.translate())


If the proteins are the same, it's a **synonymous** (silent) mutation. The DNA changed but the protein didn't.


---
## Exercise 12 — A non-synonymous mutation
Now let's look at a mutation that DOES change the protein.


In [ ]:
mutant_2 = Seq("ATGGATTCA")  # 8th base: T → C

print("Wild-type protein:", wild_type.translate())
print("Mutant 2 protein: ", mutant_2.translate())


This type of mutation (different protein) is called **non-synonymous** or **missense**.

When disease-causing mutations are found in BRCA1, most of them are non-synonymous.


---
## Exercise 13 — Write a mutation detector
Write a function that compares two DNA sequences and tells you:
1. How many DNA positions differ
2. How many amino acid positions differ
3. Whether the mutations are synonymous or not


In [ ]:
def compare_mutations(seq_a, seq_b):
    """Compare two DNA sequences and report mutations."""
    seq_a = Seq(seq_a) if isinstance(seq_a, str) else seq_a
    seq_b = Seq(seq_b) if isinstance(seq_b, str) else seq_b
    
    # DNA differences
    dna_diff = sum(1 for i in range(len(seq_a)) if seq_a[i] != seq_b[i])
    
    # Protein differences
    prot_a = seq_a.translate()
    prot_b = seq_b.translate()
    prot_diff = sum(1 for i in range(len(prot_a)) if prot_a[i] != prot_b[i])
    
    print(f"DNA positions different:     {dna_diff}")
    print(f"Protein positions different: {prot_diff}")
    if dna_diff > 0 and prot_diff == 0:
        print("→ All mutations are SYNONYMOUS (silent)")
    elif prot_diff > 0:
        print("→ At least one mutation is NON-SYNONYMOUS")
    else:
        print("→ No mutations found")

compare_mutations("ATGGATTTA", "ATGGACTTA")
print()
compare_mutations("ATGGATTTA", "ATGGATTCA")


---
## Exercise 14 — Amino acid composition
Not all amino acids are equally common in a protein. Let's count them.


In [ ]:
from collections import Counter

protein_str = str(brca1_protein)
counts = Counter(protein_str)

# Show the top 5 most common
for aa, count in counts.most_common(5):
    print(f"{aa}: {count}")


---
## Exercise 15 — Visualize amino acid composition
Build a bar chart showing how many of each amino acid are in BRCA1.


In [ ]:
import matplotlib.pyplot as plt

aa_list = sorted(counts.keys())
aa_counts = [counts[aa] for aa in aa_list]

plt.figure(figsize=(10, 4))
plt.bar(aa_list, aa_counts, color="#028090")
plt.title("Amino acid composition of BRCA1 (partial)")
plt.xlabel("Amino acid")
plt.ylabel("Count")
plt.show()


---
## Exercise 16 — Trace the insulin gene
Repeat the full DNA → RNA → Protein flow for the insulin gene.


In [ ]:
insulin = Seq("ATGGCCCTGTGGATGCGCCTCCTGCCCCTGCTGGCGCTGCTGGCCCTCTGGGGACCTGACCCAGCCGCAGCCTTTGTGAACCAACACCTGTGCGGCTCACACCTGGTGGAAGCTCTCTACCTAGTGTGCGGGGAACGAGGCTTCTTCTACACACCCAAGACC")

# Full trace in 3 lines:
rna = ?? #RNA
protein =?? #Protein

print("DNA (first 60):    ", ??)
print("RNA (first 60):    ",??)
print("Protein:           ", ??)
print("Protein length:    ", ??)


---
## Exercise 17 — 🏆 Final challenge: a real variant

Below you have two short snippets of a hypothetical gene — the **reference** (normal) version and a patient's **variant** version.

Your job: figure out what kind of mutation the patient has.


In [ ]:
reference = Seq("ATGCATCGATCGAAATAA")
patient   = Seq("ATGCATCGCTCGAAATAA")  # one base differs

# 1. Find how many DNA positions differ
dna_diff = ____

# 2. Translate both and count protein differences
ref_protein = reference.translate()
pat_protein = patient.translate()
protein_diff = ____

# 3. Print the diagnosis
print(f"DNA differences:     {dna_diff}")
print(f"Protein differences: {protein_diff}")
if protein_diff == 0:
    print("Diagnosis: SYNONYMOUS mutation (silent)")
else:
    print("Diagnosis: NON-SYNONYMOUS mutation")


### Solution
Run the cell below if you need to check your answer.


In [ ]:
reference = Seq("ATGCATCGATCGAAATAA")
patient   = Seq("ATGCATCGCTCGAAATAA")

dna_diff = sum(1 for i in range(len(reference)) if reference[i] != patient[i])
ref_protein = reference.translate()
pat_protein = patient.translate()
protein_diff = sum(1 for i in range(len(ref_protein)) if ref_protein[i] != pat_protein[i])

print(f"DNA differences:     {dna_diff}")
print(f"Protein differences: {protein_diff}")
print(f"Reference protein:   {ref_protein}")
print(f"Patient protein:     {pat_protein}")


---
## 🎉 Outstanding!

You just did what a clinical bioinformatician does every day:

- ✅ Transcribed DNA into RNA
- ✅ Translated RNA into protein
- ✅ Found start codons, stop codons, and open reading frames
- ✅ Compared a reference and a variant
- ✅ Classified mutations as synonymous or non-synonymous

### What's next?
Tomorrow (Day 2) we'll dive into **file formats** in depth (FASTQ from sequencers!), the **command line** for running real bioinformatics tools, and a proper session with **real data** from NCBI.

Great job today. See you tomorrow!
